In [1]:
import requests
import csv
import subprocess
import json
from tqdm import tqdm

In [2]:
M3U_FILE = "../tung_iptv.m3u"
OUTPUT_FILE = "channel_status.csv"

HEADERS = {
    "User-Agent": "TiviMate/5.2.0 (Linux; Android 10)"
}

TIMEOUT = 15


def parse_m3u(file_path):
    channels = []
    name = None

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if line.startswith("#EXTINF"):
                name = line.split(",")[-1]

            elif line.startswith("http"):
                channels.append({
                    "name": name,
                    "url": line
                })

    return channels


def check_stream(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT, stream=True)

        if r.status_code == 200:
            return "ONLINE"
        else:
            return f"HTTP {r.status_code}"

    except Exception:
        return "OFFLINE"


def get_quality(url):
    try:
        cmd = [
            "ffprobe",
            "-v", "quiet",
            "-print_format", "json",
            "-show_streams",
            url
        ]

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)

        data = json.loads(result.stdout)

        for stream in data.get("streams", []):
            if stream.get("codec_type") == "video":
                height = stream.get("height")

                if height >= 2160:
                    return "4K"
                elif height >= 1080:
                    return "FHD"
                elif height >= 720:
                    return "HD"
                else:
                    return "SD"

    except Exception:
        pass

    return "Unknown"

In [3]:
channels = parse_m3u(M3U_FILE)

rows = []

for ch in tqdm(channels, desc="Checking channels"):

    status = check_stream(ch["url"])

    quality = ""
    if status == "ONLINE":
        quality = get_quality(ch["url"])

    rows.append([
        ch["name"],
        ch["url"],
        status,
        quality
    ])

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)
    writer.writerow(["Channel Name", "URL", "Status", "Quality"])

    writer.writerows(rows)

print("Done. Results saved to:", OUTPUT_FILE)

Checking channels: 100%|██████████| 143/143 [13:44<00:00,  5.76s/it]

Done. Results saved to: channel_status.csv
